# 03 — Nodes 节点

**来源:** [菜鸟教程 — LangGraph 入门教程](https://www.runoob.com/ai-agent/langgraph-quick-start.html)

LangGraph 的节点是执行具体操作的函数，本 notebook 介绍各种节点类型。

## 1. 普通函数节点

节点就是普通 Python 函数，接收 state 返回部分更新。

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated


class AgentState(TypedDict):
    """状态定义"""
    messages: Annotated[list, add_messages]


def simple_node(state: AgentState) -> dict:
    """普通函数节点：读取消息并生成回复"""
    last_message = state["messages"][-1]
    response = f"收到: {last_message.content}"
    return {"messages": [{"role": "assistant", "content": response}]}


print("simple_node 定义完成")


## 2. LLM 调用节点

节点内调用 LLM 生成回复，需要先配置 `.env` 文件中的 API Key。

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os
from langchain_deepseek import ChatDeepSeek
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatDeepSeek(
    model='deepseek-v4-pro',
    temperature=0
)


def llm_node(state: dict) -> dict:
    """调用 LLM 的节点"""
    system_prompt = SystemMessage(content="你是一个有帮助的助手。")
    # 将系统提示与对话历史合并
    messages = [system_prompt] + state["messages"]
    # 调用 LLM
    response = llm.invoke(messages)
    return {"messages": [response]}


print("LLM 节点定义完成")


## 3. 异步节点

使用 `async` 定义异步节点，适合 I/O 密集型操作（API 调用、数据库查询等）。
运行异步图时使用 `graph.ainvoke()`。

In [ ]:
import asyncio


async def async_node(state: AgentState) -> dict:
    """异步节点，适合 I/O 密集型操作"""
    # 模拟异步操作
    await asyncio.sleep(0.1)
    last_msg = state["messages"][-1].content
    result = f"异步处理完成: {last_msg}"
    return {"messages": [{"role": "assistant", "content": result}]}


print("async_node 定义完成")


## 4. 使用类作为节点

类实例通过 `__call__` 方法可以作为节点使用，方便携带配置参数。

In [ ]:
from langchain_core.messages import SystemMessage


class RouterNode:
    """使用类实现的节点，可携带配置参数"""

    def __init__(self, llm, system_prompt: str):
        self.llm = llm
        self.system_prompt = system_prompt

    def __call__(self, state: AgentState) -> dict:
        """类实例可以作为节点使用"""
        messages = [
            SystemMessage(content=self.system_prompt),
            *state["messages"]
        ]
        response = self.llm.invoke(messages)
        return {"messages": [response]}


# 创建类节点实例
router = RouterNode(llm, "你是一个专业的路由助手。")
print("RouterNode 类节点定义完成")
